# MOCADB M-dwarf rotators × Gaia DR3 crossmatch

Input: `cf_data/mocadb_M_prot.csv`  
All rows already carry `GaiaDR3_ID`, so no SIMBAD step is needed.  
We query Gaia DR3 by source ID to retrieve the photometric/astrometric columns missing from the MOCADB export.

**Columns fetched from Gaia DR3:**
- `phot_g_mean_mag`, `bp_rp`, `ruwe`, `parallax` — from `gaiadr3.gaia_source`
- `ebpminrp_gspphot` — from `gaiadr3.astrophysical_parameters`
- `bprp0` computed as `bp_rp - ebpminrp_gspphot`


In [1]:
from pathlib import Path
import pandas as pd

CF_DATA = Path("cf_data")
mocadb_path = CF_DATA / "mocadb_M_prot.csv"

df_raw = pd.read_csv(mocadb_path)
print(f"Raw rows: {len(df_raw)}")
print(f"Unique moca_oid: {df_raw['moca_oid'].nunique()}")
df_raw.head()

Raw rows: 12470
Unique moca_oid: 4503


,moca_oid,GaiaDR3_ID,twomass_id,moca_aid,moca_mtid,designation,ra,dec,spectral_type,banyan_prob,gmag,rmag,bmag,parallax_mas,prot_days,age_string_myr,age_myr,age_myr_unc,age_myr_unc_pos,age_myr_unc_neg
0,254875,2993303106774038016,06150934-1501293,ABDMG,CM,Gaia DR2 2993303106774038016,93.788890,-15.025059,(M0.2),94.2004,12.611,11.610,13.623,15.313400,5.295500,133 +15/-20,133.0,12.5,15.0,10.0
1,630,306056350151904896,01472953+3413096,ABDMG,CM,G 133-40,26.873731,34.218406,(M1.0),NaN,11.448,10.405,12.554,34.221802,33.444801,133 +15/-20,133.0,12.5,15.0,10.0
2,169461,63678907415301120,03522875+2133586,ABDMG,CM,Gaia DR2 63678907415301120,58.120025,21.565811,(M1.1),99.6208,12.649,11.605,13.744,16.679800,6.865040,133 +15/-20,133.0,12.5,15.0,10.0
3,632,519087552675129984,01480159+6542222,ABDMG,CM,[SLS2012] PYC J01480+6542,27.007549,65.705953,(M1.5),NaN,12.685,11.614,13.839,17.720400,0.583400,133 +15/-20,133.0,12.5,15.0,10.0
4,11320,1996094232735807744,23242041+5442564,ABDMG,CM,[SLS2012] PYC J23243+5442,351.085637,54.715569,(M1.6),74.5930,13.004,11.926,14.181,15.689000,3.800900,133 +15/-20,133.0,12.5,15.0,10.0


## Deduplicate: one row per star

Each star can appear multiple times (once per association membership).  
Keep the row with the highest `banyan_prob` so we assign the most likely age to each star.

In [3]:
df = (
    df_raw
    .sort_values("banyan_prob", ascending=False)
    .drop_duplicates(subset="moca_oid", keep="first")
    .reset_index(drop=True)
)

print(f"Unique stars after dedup: {len(df)}")
df[["moca_oid", "designation", "moca_aid", "banyan_prob", "prot_days", "age_myr"]].head(10)

Unique stars after dedup: 4503


,moca_oid,designation,moca_aid,banyan_prob,prot_days,age_myr
0,16549,LP 415-348,CHYA,99.9999,12.22000,695.0
1,3894,LP 416-570,HYA,99.9999,15.91450,695.0
2,3820,WISEA J044406.76+190207.4,HYA,99.9999,0.37600,695.0
3,82639,Gaia DR2 3314294771901304192,HYA,99.9999,1.46000,695.0
4,82565,Gaia DR2 3311102820926310784,HYA,99.9999,3.64000,695.0
5,16537,LP 415-183,HYA,99.9999,18.44000,695.0
6,3459,[BKM2008] HB13c01-162s,HYA,99.9999,1.07100,695.0
7,3422,HG 7-244,HYA,99.9999,14.52000,695.0
8,5933,UGCS J084053.91+195955.2,CPRA,99.9999,24.42400,617.0
9,5582,2MASS J08381655+1859287,CPRA,99.9999,29.45875,617.0


## Query Gaia DR3 by source ID

In [4]:
%pip install astroquery astropy

Note: you may need to restart the kernel to use updated packages.


In [6]:
from astroquery.gaia import Gaia
import pandas as pd

def chunk_list(lst, n=500):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def query_gaia_source(source_ids):
    ids_str = ",".join(str(x) for x in source_ids)
    query = f"""
    SELECT
        source_id,
        phot_g_mean_mag,
        bp_rp,
        ruwe,
        parallax
    FROM gaiadr3.gaia_source
    WHERE source_id IN ({ids_str})
    """
    job = Gaia.launch_job(query)
    res = job.get_results()
    return res.to_pandas()

def query_gaia_ap(source_ids):
    ids_str = ",".join(str(x) for x in source_ids)
    query = f"""
    SELECT
        source_id,
        ebpminrp_gspphot
    FROM gaiadr3.astrophysical_parameters
    WHERE source_id IN ({ids_str})
    """
    job = Gaia.launch_job(query)
    res = job.get_results()
    return res.to_pandas()

In [7]:
source_ids = df["GaiaDR3_ID"].dropna().astype(int).unique().tolist()
print(f"Querying Gaia for {len(source_ids)} unique source IDs...")

gaia_core_parts = []
gaia_ap_parts = []

for i, chunk in enumerate(chunk_list(source_ids, n=500), start=1):
    print(f"Chunk {i}: {len(chunk)} IDs")
    gaia_core_parts.append(query_gaia_source(chunk))
    gaia_ap_parts.append(query_gaia_ap(chunk))

gaia_core = pd.concat(gaia_core_parts, ignore_index=True)
gaia_ap   = pd.concat(gaia_ap_parts,   ignore_index=True)

print(f"Gaia source rows returned: {len(gaia_core)}")
print(f"Gaia AP rows returned:     {len(gaia_ap)}")

Querying Gaia for 4503 unique source IDs...
Chunk 1: 500 IDs
Chunk 2: 500 IDs
Chunk 3: 500 IDs
Chunk 4: 500 IDs
Chunk 5: 500 IDs
Chunk 6: 500 IDs
Chunk 7: 500 IDs
Chunk 8: 500 IDs
Chunk 9: 500 IDs
Chunk 10: 3 IDs
Gaia source rows returned: 4503
Gaia AP rows returned:     4503


## Merge Gaia tables and compute `bprp0`

In [8]:
gaia = gaia_core.merge(gaia_ap, how="left", on="source_id")

# bprp0 = dereddened (BP-RP)
gaia["bprp0"] = gaia["bp_rp"] - gaia["ebpminrp_gspphot"]

gaia["source_id"] = gaia["source_id"].astype("Int64")
print(f"Gaia merged rows: {len(gaia)}")
gaia[["source_id", "phot_g_mean_mag", "bp_rp", "ebpminrp_gspphot", "bprp0", "ruwe", "parallax"]].head()

Gaia merged rows: 4503


,source_id,phot_g_mean_mag,bp_rp,ebpminrp_gspphot,bprp0,ruwe,parallax
0,68283421593270400,17.570240,3.414770,0.5668,2.847970,0.926502,7.416620
1,435418256850317696,16.130255,2.887178,0.5295,2.357678,1.008072,5.846047
2,436723205354481408,15.836413,2.756957,0.4521,2.304857,1.058786,5.581406
3,441573048067694592,14.862582,2.241774,0.3155,1.926274,1.040227,5.743789
4,659757283619334656,15.771846,2.354419,NaN,NaN,1.044470,5.110427


## Join Gaia data back to MOCADB

In [9]:
df["GaiaDR3_ID"] = df["GaiaDR3_ID"].astype("Int64")

df_out = df.merge(
    gaia,
    how="left",
    left_on="GaiaDR3_ID",
    right_on="source_id"
).drop(columns=["source_id"])

n_matched = df_out["bp_rp"].notna().sum()
print(f"Matched to Gaia: {n_matched} / {len(df_out)}")
print(f"Unmatched:       {len(df_out) - n_matched}")

Matched to Gaia: 4503 / 4503
Unmatched:       0


## Add RUWE flag

In [10]:
# flag_high_ruwe: True if RUWE >= 1.2 (potential binary / astrometric issue)
df_out["flag_high_ruwe"] = df_out["ruwe"].apply(
    lambda r: (r >= 1.2) if pd.notna(r) else None
)

print(f"RUWE >= 1.2 (flagged): {df_out['flag_high_ruwe'].sum()}")
print(f"RUWE <  1.2 (clean):   {(df_out['flag_high_ruwe'] == False).sum()}")
print(f"No RUWE data:          {df_out['flag_high_ruwe'].isna().sum()}")

RUWE >= 1.2 (flagged): 463
RUWE <  1.2 (clean):   4024
No RUWE data:          16


## Summary and export

In [11]:
print("=== MOCADB × Gaia DR3 crossmatch summary ===")
print(f"Total unique stars:     {len(df_out)}")
print(f"Matched to Gaia DR3:   {n_matched}")
print(f"Missing Gaia data:     {len(df_out) - n_matched}")
print(f"Flagged high RUWE:     {df_out['flag_high_ruwe'].sum()}")
print()
print(df_out[["designation", "moca_aid", "prot_days", "age_myr",
              "bp_rp", "bprp0", "ruwe", "flag_high_ruwe"]].head(10))

=== MOCADB × Gaia DR3 crossmatch summary ===
Total unique stars:     4503
Matched to Gaia DR3:   4503
Missing Gaia data:     0
Flagged high RUWE:     463

                    designation moca_aid  prot_days  age_myr     bp_rp  \
0                    LP 415-348     CHYA   12.22000    695.0  2.722661   
1                    LP 416-570      HYA   15.91450    695.0  1.981901   
2     WISEA J044406.76+190207.4      HYA    0.37600    695.0  3.216642   
3  Gaia DR2 3314294771901304192      HYA    1.46000    695.0  3.057453   
4  Gaia DR2 3311102820926310784      HYA    3.64000    695.0  2.862668   
5                    LP 415-183      HYA   18.44000    695.0  2.825480   
6        [BKM2008] HB13c01-162s      HYA    1.07100    695.0  3.543282   
7                      HG 7-244      HYA   14.52000    695.0  2.199377   
8      UGCS J084053.91+195955.2     CPRA   24.42400    617.0  2.620050   
9       2MASS J08381655+1859287     CPRA   29.45875    617.0  2.546174   

      bprp0      ruwe flag_hig

In [12]:
out_path = CF_DATA / "mocadb_gaia_xmatch.csv"
df_out.to_csv(out_path, index=False)
print(f"Saved to {out_path}")

Saved to cf_data/mocadb_gaia_xmatch.csv
